# Users Silver
---

* Cada usuário aparece **uma única vez**
* Tipos devem seguir o documento de padronização da Silver
* Datas devem seguir os tipos especificados no documento
* Dados ruins são tratados, não ignorados

---
**Bibliotecas**

* `functions (F)` &rarr; transformações declarativas (Spark SQL style)
* `Window` &rarr; deduplicação correta

---
**Leitura da Bronze**

* Estamos confiando que a Bronze:
  * já tem schema
  * já tem metadados
  * já é rastreável

---
**Seleção de colunas**

Esse é um processo simples, mas `arquiteturalmente crítico`

* Não aplicar `select("*")`:
  * Campos inesperados quebram os contratos
  * Evolução de schema fica caótica
  * Times downstream perdem confiança

> *Aqui será definido tudo de forma explícita.*

---
**Aplicação da Tipagem nas colunas**

* Aplicar as regras que estão no **data contract**
  * Não é necessário definir a tipagem a ser aplicada depois para todas as colunas, apenas para as colunas que precisam dos ajustes necessários

---
**Normalização**

Aqui será aplicado as regras definidas no documento para a normalização das nomenclaturas dos países.

* Segue a nomenclatura com 2 dígitos
* Aplicação de `upper()` 

---
**Validações Mínimas de Qualidade**

Separar os dados em tabela tratada com `user_id` não nulo;

Em outra tabela, serão armazenados os dados que não passam no processo do tratamento.

---
**Deduplicação**

> *Para cada `user_id`, manter o registro com `created_at` mais recente.*

Aqui é onde escolhemos a "verdadeira" versão de um mesmo registro quando ele aparece mais de uma vez.

**Não usar `dropDuplicates()` aqui, pois não resolve a regra.**


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

* **Leitura do dados na bronze**

In [0]:
users_bronze = "/Volumes/main/lakehouse_marketing/bronze/users/"

df_bronze = spark.read\
                .format("delta")\
                .load(users_bronze)

display(df_bronze.limit(5))
display(df_bronze.printSchema())

* **Aqui faremos:**

  * *Seleção das Colunas*
  * *Definição da Tipagem*
  * *Normalização para Country*
  * *Validações Mínimas de qualidade*


In [0]:
#########################################################
#  Seleção explícita das colunas que serão usadas
#########################################################
df = df_bronze.select(
    "user_id",
    "email",
    "country",
    "signup_date",
    "created_at",
    "ingestion_timestamp",
    "source_file"
)

#############################################################
# Isso é apenas uma forma declarativa para se aplicar depois
#############################################################
df_typed = df\
            .withColumn("email", F.lower("email"))\
            .withColumn("country", F.upper("country"))\
            .withColumn("signup_date", F.to_date("signup_date"))

#####################################
#Nomalização para a coluna country
#####################################
df_normalized = df_typed.withColumn(
    "country",
    F.when(F.col("country").isin("BRA", "BRAZIL"), "BR")
     .when(F.col("country").isin("USA", "US", "UNITED STATES"), "US")
     .otherwise("UNKNOWN")
)
display(df_normalized.groupBy("country").count())
display(df_normalized.sample(0.001))


In [0]:
df_with_rules = df_normalized.withColumn(
    "rejection_reason",
    F.when(F.col('user_id').isNull(), 'NULL_USER_ID')
     .when(F.col('email').isNull(), 'NULL_EMAIL')
     .when(~F.col('email').rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$"), "INVALID_EMAIL")
     .otherwise(None)
)

df_with_rules.show()

In [0]:
df_valid = df_with_rules.filter(F.col('rejection_reason').isNull())
df_rejected = df_with_rules.filter(F.col('rejection_reason').isNotNull())

print(df_valid.count())
print(df_rejected.count())

* Deduplicação
---

A regra: *Aplicar deduplicação em `user_id` com `created_at` mais recente*.

In [0]:
# Primeiro particionamos por user_id e ordenamos por signup_date
window = Window.partitionBy("user_id").orderBy(F.col("signup_date").desc())

# Após definimos a janela, o row_number define um rank para cada registro
# como ordenamos de forma decrescente, o maior "signup_date" de cada "user_id"
# terá o rank 1, com isso, conseguimos filtrar apenas o que for igual a  1.
df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print(f"Sem deduplicação: {df_valid.count()}")
print(f"Com deduplicação: {df_dedup.count()}")

#### Escrita na SILVER

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS  silver_marketing;
CREATE SCHEMA IF NOT EXISTS governance_marketing;

In [0]:
################################################
######## ESCRITA DOS DADOS TRATADOS ############
################################################
df_valid.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('main.lakehouse_marketing.users_bronze')


################################################
####### ESCRITA DOS DADOS SEM TRATAMENTO #######
################################################
df_rejected.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('main.governance_marketing.users_rejected')

* Verificações na tabela

In [0]:
%sql
SELECT 
    COUNT(*) AS n_registros
FROM main.lakehouse_marketing.users_bronze

In [0]:
%sql
SELECT 
    COUNT(*) AS n_registros
FROM main.governance_marketing.users_rejected